# 第五阶段：智能指针

## 实验 2：`std::unique_ptr` —— 默认的唯一所有权模型

实验 1 中，raw pointer 离开作用域不会销毁堆对象，提前返回因此跳过了 `delete`。本实验改用 `std::unique_ptr<T>`，把“唯一拥有一个动态对象，并在 owner 析构时自动释放”写进类型。

本实验关注：

- 为什么优先使用 `std::make_unique`；
- 作用域结束、提前返回时如何自动清理；
- 为什么 `unique_ptr` 不可复制但可以移动；
- 如何区分拥有、借用和显式放弃所有权；
- 为什么 `unique_ptr` 适合 C++ 内部 API，却不能直接暴露为 C ABI。

In [ ]:
// 本步骤：引入智能指针、断言、类型检查和移动语义所需的标准库。
#include <cassert>
#include <iostream>
#include <memory>
#include <type_traits>
#include <utility>

### 1. 用生命周期日志观察自动清理

`User` 的静态计数器记录当前存活对象数。后续每个示例都会确认：只要 ownership 仍由 `unique_ptr` 管理，计数最终都会回到调用前。

In [ ]:
// 本步骤：定义可观察生命周期的 User，验证 unique_ptr 何时销毁对象。
class User
{
public:
    explicit User(int id)
        : id_(id)
    {
        // 构造成功后增加计数，表示一个 User 生命周期已经开始。
        ++live_count_;
        std::cout << "User " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~User() noexcept
    {
        // 析构时减少计数，用日志确认资源清理发生在预期边界。
        --live_count_;
        std::cout << "User " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    void hello() const
    {
        // const 成员函数只读取对象，调用不会改变 ownership。
        std::cout << "hello from User " << id_ << '\n';
    }

    int id() const noexcept
    {
        // 返回普通值，不向调用方暴露内部资源。
        return id_;
    }

    static int live_count() noexcept
    {
        // 只读返回存活数量，供实验断言使用。
        return live_count_;
    }

private:
    int id_;
    inline static int live_count_ = 0;
};

### 2. 使用 `make_unique` 建立唯一所有权

默认构造的 `unique_ptr<User>` 不拥有对象，可像 nullable handle 一样用布尔条件检查。`std::make_unique<User>(...)` 创建对象并立即把 ownership 放入智能指针，避免让 owning raw pointer 暂时暴露在业务代码中。

In [ ]:
// 本步骤：从空状态建立 ownership，并观察作用域结束时自动析构。
{
    const int before = User::live_count();

    {
        // 默认构造表示当前没有拥有 User。
        std::unique_ptr<User> user;
        assert(!user);

        // make_unique 创建 User，并让 user 成为唯一 owner。
        user = std::make_unique<User>(1);
        assert(user);
        assert(User::live_count() == before + 1);

        // operator-> 访问被拥有对象，不会转移 ownership。
        user->hello();
    }

    // 离开内层作用域时 unique_ptr 析构，并自动 delete User。
    assert(User::live_count() == before);
}

所有权关系可以直接从类型读出：

```text
unique_ptr<User> owner
        │ uniquely owns
        ▼
      User

~unique_ptr() -> delete User -> ~User()
```

业务代码默认写 `std::make_unique<User>()`。只有适配要求交出 raw pointer 的底层 API、自定义 deleter 等明确场景，才应重新考虑裸 `new`。

### 3. 提前返回仍会自动清理

局部对象在函数返回时按正常的栈展开规则析构。`unique_ptr` 的析构函数负责释放它拥有的对象，因此清理不再依赖控制流是否走到函数末尾。

In [ ]:
// 本步骤：用 unique_ptr 重写实验 1 的函数，保留同样的提前返回分支。
void process_unique(bool early_return)
{
    // make_unique 把对象和 owner 的生命周期立即绑定。
    auto user = std::make_unique<User>(2);

    // return 会先析构局部变量 user，因此不会跳过对象清理。
    if (early_return)
    {
        std::cout << "early return: unique_ptr still cleans up\n";
        return;
    }

    // 正常路径只使用对象，不需要在末尾手写 delete。
    user->hello();
}

In [ ]:
// 本步骤：对比正常返回与提前返回，确认两条路径都会恢复存活计数。
{
    const int before = User::live_count();

    // 正常路径在函数结束时自动释放 User。
    process_unique(false);
    assert(User::live_count() == before);

    // 提前返回路径同样会析构局部 unique_ptr。
    process_unique(true);
    assert(User::live_count() == before);
}

### 4. 不可复制，但可以移动

若两个 `unique_ptr` 能复制并同时声称拥有同一对象，就会在析构时重复释放。因此复制操作被类型系统禁止；移动则把 ownership 从源对象转移到目标对象，转移后源指针为空。

In [ ]:
// 本步骤：在编译期确认 unique_ptr 的复制和移动能力。
static_assert(!std::is_copy_constructible_v<std::unique_ptr<User>>);
static_assert(!std::is_copy_assignable_v<std::unique_ptr<User>>);
static_assert(std::is_move_constructible_v<std::unique_ptr<User>>);
static_assert(std::is_move_assignable_v<std::unique_ptr<User>>);

// 反例只用于阅读：复制会导致编译错误，因为不能出现两个唯一 owner。
// auto first = std::make_unique<User>(3);
// auto second = first;

In [ ]:
// 本步骤：移动 unique_ptr，并观察 owner 改变而 User 对象保持不变。
{
    const int before = User::live_count();

    {
        auto source = std::make_unique<User>(3);

        // get() 取得同一对象的非拥有地址，仅用于确认移动前后对象身份。
        User *observer = source.get();

        // std::move 将 ownership 转移给 destination，不复制 User。
        std::unique_ptr<User> destination = std::move(source);

        // 源 owner 变空，目标 owner 仍指向原来的 User。
        assert(!source);
        assert(destination.get() == observer);
        assert(User::live_count() == before + 1);
        destination->hello();
    }

    // 最终 owner 离开作用域后，对象只被销毁一次。
    assert(User::live_count() == before);
}

### 5. owner 可以临时提供借用

多数只使用对象的函数不需要接收 `unique_ptr`。传 `const User&` 表示同步只读借用，调用方继续持有 ownership；`owner.get()` 返回的 `User*` 同样不拥有对象，不能 `delete`，也不能活得比 owner 更久。

In [ ]:
// 本步骤：定义只读借用接口；参数类型没有取得或共享 ownership。
void inspect_user(const User &user)
{
    // 引用只在调用期间访问现有 User。
    std::cout << "inspect User " << user.id() << '\n';
}

In [ ]:
// 本步骤：从 unique_ptr 提供引用和 raw pointer 借用，不改变 owner。
{
    auto owner = std::make_unique<User>(4);

    // 解引用 owner 得到 User&，只在本次调用期间借用。
    inspect_user(*owner);

    // get() 返回观察指针；owner 仍然非空并负责最终销毁。
    User *observer = owner.get();
    assert(observer == owner.get());
    observer->hello();

    // observer 在 owner 析构或 reset 后就会悬空，因此不能保存到更长生命周期。
}

### 6. 通过返回值转移 ownership

返回 `std::unique_ptr<User>` 表示调用方将成为新 owner。返回时可以发生移动或返回值优化，不会复制底层 `User`。这是一种清晰的 C++ 工厂接口。

In [ ]:
// 本步骤：定义工厂函数，用返回类型声明 ownership 交给调用方。
std::unique_ptr<User> create_user(int id)
{
    // make_unique 创建的 owner 作为返回值转移出去。
    return std::make_unique<User>(id);
}

In [ ]:
// 本步骤：接收工厂返回值，确认调用方成为对象的新 owner。
{
    const int before = User::live_count();

    {
        // 返回类型明确要求 receiver 保存或继续转移 ownership。
        std::unique_ptr<User> user = create_user(5);
        assert(user && user->id() == 5);
        assert(User::live_count() == before + 1);
    }

    // 调用方的 unique_ptr 析构后，工厂创建的对象也被释放。
    assert(User::live_count() == before);
}

### 7. `reset()` 与 `release()` 的边界

- `reset()` 销毁当前对象并让指针变空；给 owner 移动赋值会销毁旧对象并接管右侧的新对象。
- `release()` 只交出地址，不销毁对象。返回值重新成为 owning raw pointer，调用方必须安排一次释放。

普通业务代码很少需要 `release()`；它主要用于把 ownership 交给明确承诺接管资源的底层 API。

In [ ]:
// 本步骤：观察重新赋值和 reset 都会自动结束原对象生命周期。
{
    const int before = User::live_count();
    auto owner = std::make_unique<User>(6);

    // 右侧先创建 User 7；随后移动赋值销毁 User 6，并接管 User 7。
    owner = std::make_unique<User>(7);
    assert(owner->id() == 7);
    assert(User::live_count() == before + 1);

    // reset 销毁 User 7，并把 owner 切换为空状态。
    owner.reset();
    assert(!owner);
    assert(User::live_count() == before);
}

In [ ]:
// 本步骤：受控演示 release，确认它会放弃自动清理责任。
{
    const int before = User::live_count();
    auto owner = std::make_unique<User>(8);

    // release 后 raw_owner 取得 ownership，原 unique_ptr 变空且不会 delete。
    User *raw_owner = owner.release();
    assert(!owner);
    assert(User::live_count() == before + 1);

    // 实验中没有真正的接管 API，因此必须手动完成唯一一次 delete。
    delete raw_owner;
    raw_owner = nullptr;
    assert(User::live_count() == before);
}

### 8. C++ API 与 ABI 边界

下面两个 C++ 内部接口直接表达 ownership contract：

```cpp
std::unique_ptr<User> create_user();       // 返回唯一 ownership
void consume(std::unique_ptr<User> user);  // 接收并消费 ownership
void inspect(const User &user);            // 仅同步借用
```

但 `std::unique_ptr` 是依赖 C++ 构造、析构、模板实例化和标准库 ABI 的类型，不能直接暴露给 C 或 Kotlin/Native。跨边界时通常改为 opaque handle，并由同一原生库提供成对函数：

```text
C++ Core                         C ABI / Kotlin Native
unique_ptr<User> owner   ->      sdk_user_handle*
自动析构                  ->      sdk_user_destroy(handle)
```

Kotlin 包装层再把 handle 放进可关闭对象，并确保最终调用原生 destroy。不能把 `owner.get()` 得到的临时借用误当成可长期保存的跨语言 handle。

## 实验结论

| 表达式 | ownership 语义 | 结果 |
| --- | --- | --- |
| `std::make_unique<User>()` | 创建唯一 owner | owner 析构时自动释放 |
| `std::move(source)` | 转移 ownership | `source` 变空 |
| `*owner` / `owner.get()` | 临时借用 | 不得释放，不得超过 owner 生命周期 |
| `owner.reset()` | 结束 ownership | 当前对象立即销毁 |
| `owner.release()` | 放弃自动管理 | 调用方重新承担释放责任 |
| 返回 `unique_ptr<T>` | 把 ownership 交给调用方 | 适合 C++ 工厂 API |

看到 `std::unique_ptr<User>`，应该直接读成：**这里恰好有一个 owner；不能复制 ownership，只能移动；owner 生命周期结束时自动销毁 User。**

这就是实验 1 的修复原则：不要靠每条控制流记住 `delete`，而要把资源所有权绑定到 RAII 对象的生命周期。